In [1]:
# NOTEBOOK 37 - deployable score: the best wind-aware model (dyn_adv) when the decoder is shown the REAL day-old forecast wind, not the actual future wind
# Why: the 57.71 of notebook 28 used ERA5 wind for the 24 FUTURE hours (a perfect forecast, impossible in a live system). A live system only has a forecast.
# Notebook 19/20 showed that a real day-old forecast recovers almost all of the benefit for the plain models; this runs the same test for the transport-feature models.
# What changes versus notebook 28: ONLY the future-hour inputs of the decoder - the 3 wind channels (u, v, speed) and the forecast graph A_fc come from the
# day-old forecast (openmeteo_prevday1_uv.npy / dynamic_adj_prevday1.npy). The 48 past hours still use observed (ERA5) wind. Same features, loss, seeds and split.
# Variants (5 seeds each): dyn_adv_fc (wind graph + upwind feature), none_adv_fc (graph-free + upwind feature), none_nb_fc (graph-free + neighbour mean).
# Also: (a) a quick check of what happens if the existing oracle-trained dyn_adv models are simply fed the forecast wind, and
#       (b) a simple average of the three new variants (chosen on validation, one extra row, not a replacement).
# New checkpoints: watgcn_v7<variant>_seed*.pt. Results: results/v7_realfc_delhi.json. Nothing existing is overwritten. About 5 minutes per model, 15 models, ~75 min.
# HOW TO CHECK: this cell only prints a note.
print('deployable-score test: dyn_adv_fc, none_adv_fc, none_nb_fc (seeds 42/7/123/2024/999), real day-old forecast wind for the 24 future hours')

deployable-score test: dyn_adv_fc, none_adv_fc, none_nb_fc (seeds 42/7/123/2024/999), real day-old forecast wind for the 24 future hours


In [2]:
import os, json, time, copy, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
OUT    = os.path.join(V2_DIR, 'ecotwin_outputs')
SCALER_DIR, PLOT_DIR   = os.path.join(OUT, 'scalers'), os.path.join(OUT, 'plots')
MODEL_DIR, RESULTS_DIR = os.path.join(OUT, 'models'),  os.path.join(OUT, 'results')

DATASET_CSV     = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
STATIONS_EXCEL  = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')
ERA5_WIND_NPY   = os.path.join(V2_DIR, 'openmeteo_reanalysis_uv.npy')
PREVDAY_NPY     = os.path.join(V2_DIR, 'openmeteo_prevday1_uv.npy')
ADJ_FC_NPY      = os.path.join(V2_DIR, 'dynamic_adj_prevday1.npy')
DYN_ADJ_OM_NPY  = os.path.join(V2_DIR, 'dynamic_adj_om.npy')
STATIC_ADJ_NPY  = os.path.join(V2_DIR, 'static_adj_matrix.npy')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_WINDOW, HORIZON = 48, 24
BATCH_SIZE = 16
LOSS_VOL_ALPHA = 1.0
SEEDS = [42, 7, 123, 2024, 999]

cfg = json.load(open(os.path.join(MODEL_DIR, 'v3_config.json')))
FEATURE_COLS = cfg['feature_cols']; F = cfg['in_features']
HIDDEN_DIM, GRU_LAYERS = cfg['hidden_dim'], cfg['gru_layers']
ATTN_HEADS, DROPOUT, GCN_HOPS = cfg['attn_heads'], cfg['dropout'], cfg['gcn_hops']
N_FUT = cfg['n_fut']
print(f'Device {DEVICE} | F={F} | n_fut={N_FUT} | seeds {SEEDS}')

Device cuda | F=26 | n_fut=10 | seeds [42, 7, 123, 2024, 999]


In [3]:
df = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp'])
df['Station'] = df['Station'].str.strip()
df = df.sort_values(['Station', 'Timestamp']).reset_index(drop=True)
df_st = pd.read_excel(STATIONS_EXCEL)
df_st.columns = df_st.columns.str.strip().str.lower()
df_st['station'] = df_st['station'].str.strip()
station_names = df_st['station'].tolist()
N = len(station_names)

timestamps = np.array(sorted(df['Timestamp'].unique())); T = len(timestamps)
era5 = np.load(ERA5_WIND_NPY).astype(np.float32)
era5_spd = np.hypot(era5[..., 0], era5[..., 1])
dyn_adj = np.load(DYN_ADJ_OM_NPY).astype(np.float32)                 # A(t) from ERA5 wind (T,N,N)
static_adj = np.load(STATIC_ADJ_NPY).astype(np.float32)
np.fill_diagonal(static_adj, 0.0)
assert dyn_adj.shape == (T, N, N)
print(f'T={T}  dyn_adj(ERA5) {dyn_adj.shape}  static edges {int((static_adj>0).sum())}')
prevday = np.load(PREVDAY_NPY).astype(np.float32)                    # day-old forecast wind (T,N,2), blowing TOWARD
adj_fc = np.load(ADJ_FC_NPY).astype(np.float32)                      # forecast-wind graph for the future hours (T,N,N)
assert prevday.shape == era5.shape and adj_fc.shape == dyn_adj.shape, (prevday.shape, adj_fc.shape)
print('forecast wind', prevday.shape, '| forecast graph', adj_fc.shape, '| mean |forecast - ERA5| wind speed difference (m/s):',
      float(np.abs(np.hypot(prevday[..., 0], prevday[..., 1]) - era5_spd).mean()))

T=19921  dyn_adj(ERA5) (19921, 37, 37)  static edges 508
forecast wind (19921, 37, 2) | forecast graph (19921, 37, 37) | mean |forecast - ERA5| wind speed difference (m/s): 0.7865347862243652


In [4]:
BASE_FEATS = ['pm2.5','pm10','no2','so2','co','o3','rh','at']
g = df.groupby('Station', sort=False)
df['pm_roll6_mean']  = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).mean())
df['pm_roll6_std']   = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).std()).fillna(0.0)
df['pm_roll24_mean'] = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).mean())
df['pm_roll24_std']  = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).std()).fillna(0.0)
df['pm_delta3']      = g['pm2.5'].transform(lambda s: s - s.shift(3)).fillna(0.0)
tsg = df.groupby('Timestamp', sort=False)['pm2.5']
df['city_mean_pm']  = tsg.transform('mean')
df['city_max_pm']   = tsg.transform('max')
df['pm_minus_city'] = df['pm2.5'] - df['city_mean_pm']
hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month
df['hour_sin'] = np.sin(2*np.pi*hh/24);      df['hour_cos'] = np.cos(2*np.pi*hh/24)
df['dow_sin']  = np.sin(2*np.pi*dow/7);      df['dow_cos']  = np.cos(2*np.pi*dow/7)
df['mon_sin']  = np.sin(2*np.pi*(mon-1)/12); df['mon_cos']  = np.cos(2*np.pi*(mon-1)/12)
df['is_weekend'] = (dow >= 5).astype(float)

CPCB_COLS = (BASE_FEATS +
    ['pm_roll6_mean','pm_roll6_std','pm_roll24_mean','pm_roll24_std','pm_delta3',
     'city_mean_pm','city_max_pm','pm_minus_city',
     'hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend'])
Xc = np.zeros((T, N, len(CPCB_COLS)), dtype=np.float32)
for fi, col in enumerate(CPCB_COLS):
    wide = df.pivot_table(index='Timestamp', columns='Station', values=col).reindex(columns=station_names)
    Xc[:, :, fi] = np.nan_to_num(wide.values.astype(np.float32))

built = (BASE_FEATS + ['u_wind', 'v_wind', 'wspd'] + CPCB_COLS[len(BASE_FEATS):])
assert built == FEATURE_COLS, 'feature order mismatch vs v3_config'
X_raw = np.concatenate([
    Xc[:, :, :len(BASE_FEATS)],
    era5[:, :, 0:1], era5[:, :, 1:2], era5_spd[:, :, None],
    Xc[:, :, len(BASE_FEATS):],
], axis=2).astype(np.float32)
Y_raw = X_raw[:, :, 0].copy()

TIME_COLS = ['hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend']
TIME_IDX  = [FEATURE_COLS.index(c) for c in TIME_COLS]
U_IDX, V_IDX, W_IDX = FEATURE_COLS.index('u_wind'), FEATURE_COLS.index('v_wind'), FEATURE_COLS.index('wspd')
FUT_IDX   = TIME_IDX + [U_IDX, V_IDX, W_IDX]
print(f'F={X_raw.shape[2]}  NaNs {int(np.isnan(X_raw).sum())}')

C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_20488\1524811802.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month


F=26  NaNs 0


In [5]:
# 1) geometry and the two new feature pairs for Delhi
lats = df_st['latitude'].values.astype(float); lons = df_st['longitude'].values.astype(float)
THRESHOLD_KM, MAX_LAG = 12.0, 6

def geometry(lats, lons):
    n = len(lats)
    dist = np.zeros((n, n)); du = np.zeros((n, n)); dv = np.zeros((n, n))
    R = 6371.0
    for i in range(n):
        for j in range(n):
            if i != j:
                p1, p2 = np.radians(lats[i]), np.radians(lats[j])
                a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lons[j] - lons[i]) / 2) ** 2
                dist[i, j] = 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
                de = (lons[j] - lons[i]) * np.cos(np.radians(lats[i])); dn = lats[j] - lats[i]; nr = np.hypot(de, dn)
                if nr > 0:
                    du[i, j], dv[i, j] = de / nr, dn / nr          # unit vector from i to j
    return dist, du, dv, (dist > 0) & (dist <= THRESHOLD_KM)

def transport_features(pm, uv, dist, du, dv, within):
    # pm (T,n) raw PM2.5, uv (T,n,2) ERA5 wind (blowing TOWARD). Only values at or before hour t are used (no leakage).
    T, n = pm.shape
    adv = pm.copy(); nb = pm.copy()
    src = np.arange(n)[:, None] * np.ones((1, n), dtype=int)           # source station index j for entry [j, i]
    stat = within.astype(float); cnt = stat.sum(0); has_nb = cnt > 0
    for t in range(T):
        align = uv[t, :, 0][:, None] * du + uv[t, :, 1][:, None] * dv   # wind at j along the direction j -> i  (m/s)
        w = np.where(within & (align > 0), align / (dist + 1e-6), 0.0)  # j is upwind of i
        wsum = w.sum(0); has = wsum > 0
        lag = np.clip(np.rint(dist / (np.maximum(align, 0.5) * 3.6)), 0, MAX_LAG).astype(int)   # hours of travel (km / (km/h))
        vals = pm[np.clip(t - lag, 0, T - 1), src]                      # pm[t - lag[j,i], j]
        adv[t, has] = ((w * vals).sum(0))[has] / wsum[has]
        nb[t, has_nb] = ((stat * pm[t][:, None]).sum(0))[has_nb] / cnt[has_nb]
    return adv, nb

dist, du, dv, within = geometry(lats, lons)
t0 = time.time()
adv, nb = transport_features(X_raw[:, :, 0].astype(float), era5, dist, du, dv, within)
pm0 = X_raw[:, :, 0]
extra = np.stack([adv - 0, adv - pm0, nb - 0, nb - pm0], axis=2).astype(np.float32)       # adv_pm, adv_diff, nb_pm, nb_diff
extra[:, :, 0] = adv; extra[:, :, 2] = nb
X30 = np.concatenate([X_raw, extra], axis=2)
print(f'features built in {time.time()-t0:.0f}s | X30 {X30.shape}')
print('mean |adv - own| :', float(np.abs(adv - pm0).mean()), '| mean |nb - own| :', float(np.abs(nb - pm0).mean()))
print('corr(adv, own PM):', float(np.corrcoef(adv.ravel(), pm0.ravel())[0, 1]), '| corr(nb, own PM):', float(np.corrcoef(nb.ravel(), pm0.ravel())[0, 1]))

# HOW TO CHECK: about a minute. Both correlations are high (0.8-0.95) because neighbouring stations resemble each other;
# the question for the models is whether the lagged, wind-aligned version adds anything beyond the same-hour mean.

features built in 1s | X30 (19921, 37, 30)
mean |adv - own| : 28.104649143919474 | mean |nb - own| : 23.36611597159713
corr(adv, own PM): 0.8577609401778713 | corr(nb, own PM): 0.9041709851392609


In [6]:
# 2) scaler for the 30 channels (fit on the training period only); the first 26 must match the saved v3 scaler
from sklearn.preprocessing import StandardScaler
import pickle
T = X30.shape[0]; tr_end = int(T * 0.70)
sc30 = StandardScaler().fit(X30[:tr_end].reshape(-1, 30))
v3s = pickle.load(open(os.path.join(SCALER_DIR, 'v3_feature_scaler.pkl'), 'rb'))
assert np.allclose(sc30.mean_[:26], v3s.mean_, rtol=1e-4, atol=1e-4) and np.allclose(sc30.scale_[:26], v3s.scale_, rtol=1e-4, atol=1e-4), 'scaler mismatch'
log_scaler = pickle.load(open(os.path.join(SCALER_DIR, 'v3_log_scaler.pkl'), 'rb'))
def from_log(a): return np.expm1(log_scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape))
def to_log(a):   return log_scaler.transform(np.log1p(np.clip(a, 0, None)).reshape(-1, 1)).reshape(a.shape).astype(np.float32)
Xs_full = sc30.transform(X30.reshape(-1, 30)).reshape(T, N, 30).astype(np.float32)
Yl_full = to_log(X30[:, :, 0])
Y_raw = X30[:, :, 0].copy()
print('scaler check passed: the first 26 channels match the v3 scaler')

# HOW TO CHECK: prints the pass line. An assertion error means the data or train split changed - stop and tell me.

scaler check passed: the first 26 channels match the v3 scaler


C:\Users\jlakk_j7eh5fd\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
# future-hour wind channels from the day-old forecast (scaled with the same train-period scaler as the observed wind)
pv_spd = np.hypot(prevday[..., 0], prevday[..., 1])
pv_raw = np.stack([prevday[..., 0], prevday[..., 1], pv_spd], axis=2).astype(np.float32)          # (T,N,3)
WIDX = [U_IDX, V_IDX, W_IDX]
Xs_fut = Xs_full.copy()
Xs_fut[:, :, WIDX] = ((pv_raw - sc30.mean_[WIDX]) / sc30.scale_[WIDX]).astype(np.float32)
print('future wind channels now come from the day-old forecast | scaled means: observed', Xs_full[:, :, WIDX].mean((0, 1)).round(2), 'forecast', Xs_fut[:, :, WIDX].mean((0, 1)).round(2))

# HOW TO CHECK: the u and v means are close; the forecast speed mean is lower than the observed one (about -0.5 against -0.1 on the scaled axis) because the day-old forecast runs slower than ERA5. That bias is real and is why the models are retrained on it.

future wind channels now come from the day-old forecast | scaled means: observed [ 0.04  0.01 -0.13] forecast [-0.07  0.14 -0.5 ]


In [8]:
# 3) windows, datasets (each model picks its own columns) and loaders
BASE_COLS = list(range(26)); ADV_COLS = BASE_COLS + [26, 27]; NB_COLS = BASE_COLS + [28, 29]
COLS = {'base': BASE_COLS, 'adv': ADV_COLS, 'nb': NB_COLS}
val_end = int(T * 0.85)
def split_starts(lo, hi): return np.arange(lo, lo + (hi - lo) - (INPUT_WINDOW + HORIZON) + 1)
train_starts, val_starts, test_starts = split_starts(0, tr_end), split_starts(tr_end, val_end), split_starts(val_end, T)
print(f'windows -- train {len(train_starts):,} | val {len(val_starts):,} | test {len(test_starts):,}')

class DS(Dataset):
    def __init__(self, starts, cols, weights=None):
        self.s = np.asarray(starts, np.int64); self.cols = cols
        self.w = (weights if weights is not None else np.ones(len(self.s), np.float32)).astype(np.float32)
    def __len__(self): return len(self.s)
    def __getitem__(self, k):
        g = int(self.s[k]); i1 = g + INPUT_WINDOW; i2 = i1 + HORIZON
        return (torch.from_numpy(np.ascontiguousarray(Xs_full[g:i1][:, :, self.cols])), torch.from_numpy(dyn_adj[g:i1].copy()),
                torch.from_numpy(adj_fc[i1:i2].copy()), torch.from_numpy(np.ascontiguousarray(Xs_fut[i1:i2][:, :, FUT_IDX])),
                torch.from_numpy(Yl_full[i1-1].copy()), torch.from_numpy(Yl_full[i1:i2].copy()), torch.tensor(self.w[k], dtype=torch.float32))
sw = np.array([(Y_raw[g+INPUT_WINDOW:g+INPUT_WINDOW+HORIZON].max(0) - Y_raw[g+INPUT_WINDOW:g+INPUT_WINDOW+HORIZON].min(0)).mean() for g in train_starts], dtype=np.float32)
tr_w = np.clip(1.0 + LOSS_VOL_ALPHA * (sw / (np.median(sw) + 1e-6)), 0.5, 4.0).astype(np.float32); tr_w *= len(tr_w) / tr_w.sum()
LD = {fs: {'train': DataLoader(DS(train_starts, cols, tr_w), batch_size=BATCH_SIZE, shuffle=True, num_workers=0),
           'val':   DataLoader(DS(val_starts, cols),  batch_size=BATCH_SIZE, shuffle=False, num_workers=0),
           'test':  DataLoader(DS(test_starts, cols), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)} for fs, cols in COLS.items()}
print('loaders ready:', {fs: len(v['train']) for fs, v in LD.items()}, 'train batches')

# HOW TO CHECK: windows 13873 / 2917 / 2918, 868 train batches per feature set.

windows -- train 13,873 | val 2,917 | test 2,918
loaders ready: {'base': 868, 'adv': 868, 'nb': 868} train batches


In [9]:
# model with three graph modes plus the learned blend, and optional context dropout at training time
CTX_COLS = ['city_mean_pm', 'city_max_pm', 'pm_minus_city']
CTX_IDX = [FEATURE_COLS.index(c) for c in CTX_COLS]
WSPD_IDX = FEATURE_COLS.index('wspd')
FUT_WSPD = 9                                  # wspd is the last of the 10 future channels (7 calendar + u, v, wspd)

class WATGCNv4(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads, forecast_horizon, n_stations,
                 dropout=0.2, n_fut=10, gcn_hops=2, graph_mode='dynamic', static_adj=None):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(in_features, H)
        self.gcn_lin  = nn.ModuleList(nn.Linear(H, H) for _ in range(gcn_hops))
        self.gcn_norm = nn.ModuleList(nn.LayerNorm(H) for _ in range(gcn_hops))
        self.act = nn.ReLU()
        self.gru = nn.GRU(H, H, gru_layers, batch_first=True, dropout=dropout if gru_layers > 1 else 0.0)
        self.attn = nn.MultiheadAttention(H, attn_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        self.fut_enc = nn.Sequential(nn.Linear(n_fut, H), nn.ReLU(), nn.Linear(H, H))
        self.decoder = nn.Sequential(nn.Linear(H*3, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH, self.graph_mode = H, forecast_horizon, graph_mode
        sa = torch.eye(n_stations) if static_adj is None else torch.from_numpy(static_adj.astype('float32'))
        sa = sa.clone(); sa.fill_diagonal_(0)
        self.register_buffer('static_adj', sa, persistent=False)
        self.register_buffer('static_rn', sa / sa.sum(1, keepdim=True).clamp(min=1.0), persistent=False)   # row-normalised
        if graph_mode == 'blend':
            self.blend_a = nn.Parameter(torch.zeros(1)); self.blend_b = nn.Parameter(torch.zeros(1))

    @staticmethod
    def _n(A):
        return A / A.sum(-1, keepdim=True).clamp(min=1e-6)

    def alpha(self, speed):                       # speed: (B, steps) standardised wind speed -> weight of the wind graph
        return torch.sigmoid(self.blend_a + self.blend_b * speed)[..., None, None]

    def forward(self, X, At, Afc, Fut, Anchor):
        B, Tin, n, _ = X.shape; dev = X.device; I = torch.eye(n, device=dev)
        h = self.act(self.input_proj(X))
        if self.graph_mode == 'none':
            A = None
        elif self.graph_mode == 'dynamic':
            A = self._n(At + I.view(1, 1, n, n))
        elif self.graph_mode == 'static':
            A = self._n((self.static_adj + I).view(1, 1, n, n)).expand(B, Tin, n, n)
        else:                                      # blend
            a = self.alpha(X[..., WSPD_IDX].mean(-1))
            A = self._n(a * At + (1 - a) * self.static_rn.view(1, 1, n, n) + I.view(1, 1, n, n))
        for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
            h = self.act(nrm(lin(h if A is None else torch.matmul(A, h)))) + h
        hf = h.permute(0, 2, 1, 3).reshape(B*n, Tin, self.H)
        gg, _ = self.gru(hf); gg = self.dropout(gg)
        aa, _ = self.attn(gg, gg, gg)
        ctx = self.attn_norm(aa + gg)[:, -1, :].reshape(B, n, self.H)
        ctx_rep = ctx.unsqueeze(1).expand(B, self.FH, n, self.H)
        if self.graph_mode == 'none':
            ctx_fc = ctx_rep
        elif self.graph_mode == 'dynamic':
            ctx_fc = torch.matmul(self._n(Afc + I.view(1, 1, n, n)), ctx_rep)
        elif self.graph_mode == 'static':
            ctx_fc = torch.matmul(self._n((self.static_adj + I).view(1, 1, n, n)), ctx_rep)
        else:
            af = self.alpha(Fut[..., FUT_WSPD].mean(-1))
            ctx_fc = torch.matmul(self._n(af * Afc + (1 - af) * self.static_rn.view(1, 1, n, n) + I.view(1, 1, n, n)), ctx_rep)
        resid = self.decoder(torch.cat([ctx_rep, ctx_fc, self.fut_enc(Fut)], dim=-1)).squeeze(-1)
        return resid + Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)

In [10]:
def weighted_huber(pred, target, w, delta=1.0):
    absr = (pred - target).abs(); quad = torch.clamp(absr, max=delta)
    per = (0.5*quad**2 + delta*(absr - quad)).mean(dim=(1, 2))
    return (per * w).mean() / w.mean().clamp(min=1e-6)

def run_epoch(model, loader, opt=None, sched=None):
    tr = opt is not None; model.train(tr); tot = n = 0
    for X, At, Afc, Fut, anc, y, w in loader:
        X, At, Afc = X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE)
        Fut, anc, y, w = Fut.to(DEVICE), anc.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
        with torch.set_grad_enabled(tr):
            out = model(X, At, Afc, Fut, anc)
            loss = weighted_huber(out, y, w if tr else torch.ones_like(w))
        if tr:
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if sched: sched.step()
        tot += loss.item() * X.size(0); n += X.size(0)
    return tot / n

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, A = [], []
    for X, At, Afc, Fut, anc, y, w in loader:
        P.append(model(X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE), Fut.to(DEVICE), anc.to(DEVICE)).cpu().numpy())
        A.append(y.numpy())
    return np.concatenate(P), np.concatenate(A)

def sc(pl, al):
    pr, ar = from_log(pl), from_log(al)
    return dict(rmse=float(np.sqrt(mean_squared_error(ar.ravel(), pr.ravel()))),
                mae=float(mean_absolute_error(ar.ravel(), pr.ravel())),
                r2=float(r2_score(ar.ravel(), pr.ravel())))

In [11]:
# 4) variants, training and progress line
STATIC_NP = np.load(STATIC_ADJ_NPY).astype(np.float32)
BASE = {}
NEW = {'dyn_adv_fc': ('dynamic', 'adv'), 'none_adv_fc': ('none', 'adv'), 'none_nb_fc': ('none', 'nb')}
SPEC = {**BASE, **NEW}
def build_t(name, n=None, static_adj=None):
    mode, fs = SPEC[name]
    return WATGCNv4(len(COLS[fs]), HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, n or N, DROPOUT, N_FUT, GCN_HOPS,
                    graph_mode=mode, static_adj=STATIC_NP if static_adj is None else static_adj).to(DEVICE)
def ckpt_t(name, sd):
    base = {'dynamic': f'watgcn_v3_seed{sd}.pt', 'static': f'watgcn_v3static_seed{sd}.pt', 'none': f'watgcn_v3none_seed{sd}.pt'}
    return os.path.join(MODEL_DIR, f'watgcn_v7{name}_seed{sd}.pt')

_prog = {'done': 0, 'total': 0, 't0': time.time(), 'times': []}
def _progress(name, seed, ep, vl, best):
    el = (time.time() - _prog['t0']) / 60
    left = (np.mean(_prog['times']) * (_prog['total'] - _prog['done'] - 1)) if _prog['times'] else None
    line = (f"[model {_prog['done'] + 1}/{_prog['total']}] {name} seed {seed} | epoch {ep} | val {vl:.4f} (best {best:.4f}) | "
            f"elapsed {el:.0f} min" + (f" | ~{left:.0f} min left after this model" if left is not None else ''))
    print(line + ' ' * 12, end='\r', flush=True)

def train_t(name, seed, max_epochs=60, patience=12, lr=1e-3):
    fs = SPEC[name][1]; tr, va = LD[fs]['train'], LD[fs]['val']
    torch.manual_seed(seed); np.random.seed(seed)
    model = build_t(name)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = len(tr) * max_epochs; warm = max(1, int(0.05 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/warm if s < warm else 0.5*(1 + np.cos(np.pi*(s-warm)/max(1, steps-warm))))
    best, best_state, ctr = 1e9, None, 0
    for ep in range(1, max_epochs + 1):
        trl = run_epoch(model, tr, opt, sched); vl = run_epoch(model, va)
        if vl < best - 1e-5: best, best_state, ctr = vl, copy.deepcopy(model.state_dict()), 0
        else: ctr += 1
        _progress(name, seed, ep, vl, best)
        if ctr >= patience: break
    model.load_state_dict(best_state)
    return model

ALL = list(BASE) + list(NEW)
_todo = [(n_, s_) for n_ in ALL for s_ in SEEDS if not os.path.exists(ckpt_t(n_, s_))]
_prog['total'] = len(_todo); _prog['t0'] = time.time()
print(f'{len(_todo)} models to train, {len(ALL) * len(SEEDS) - len(_todo)} already saved')
pred = {'val': {}, 'test': {}}
for name in ALL:
    fs = SPEC[name][1]; pred['val'][name], pred['test'][name] = [], []
    for sd in SEEDS:
        f = ckpt_t(name, sd)
        if os.path.exists(f):
            m = build_t(name); m.load_state_dict(torch.load(f, map_location=DEVICE)); tag = 'loaded'
        else:
            _t = time.time(); m = train_t(name, sd); torch.save(m.state_dict(), f); tag = 'trained'
            _prog['times'].append((time.time() - _t) / 60); _prog['done'] += 1; print()
        for split in ('val', 'test'):
            p, a = predict(m, LD[fs][split]); pred[split][name].append(p)
            if split == 'val': val_actual = a
            else: test_actual = a
        print(f'[{name}] seed {sd} {tag}')
        del m; torch.cuda.empty_cache()

# HOW TO CHECK: about 100 minutes the first time. If it stops, run this cell again: finished seeds are loaded.

15 models to train, 0 already saved
[model 1/15] dyn_adv_fc seed 42 | epoch 15 | val 0.1470 (best 0.1347) | elapsed 5 min            
[dyn_adv_fc] seed 42 trained
[model 2/15] dyn_adv_fc seed 7 | epoch 15 | val 0.1458 (best 0.1347) | elapsed 11 min | ~67 min left after this model            
[dyn_adv_fc] seed 7 trained
[model 3/15] dyn_adv_fc seed 123 | epoch 17 | val 0.1553 (best 0.1412) | elapsed 16 min | ~63 min left after this model            
[dyn_adv_fc] seed 123 trained
[model 4/15] dyn_adv_fc seed 2024 | epoch 15 | val 0.1487 (best 0.1367) | elapsed 22 min | ~59 min left after this model            
[dyn_adv_fc] seed 2024 trained
[model 5/15] dyn_adv_fc seed 999 | epoch 14 | val 0.1532 (best 0.1384) | elapsed 26 min | ~53 min left after this model            
[dyn_adv_fc] seed 999 trained
[model 6/15] none_adv_fc seed 42 | epoch 15 | val 0.1588 (best 0.1348) | elapsed 31 min | ~47 min left after this model            
[none_adv_fc] seed 42 trained
[model 7/15] none_adv_fc seed

In [12]:
# check (a): take the EXISTING dyn_adv models (trained with the perfect ERA5 future wind) and simply feed them the day-old forecast wind instead
SPEC['dyn_adv_oracle'] = ('dynamic', 'adv')
ps_or = []
for sd in SEEDS:
    m = build_t('dyn_adv_oracle'); m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'watgcn_v5dyn_adv_seed{sd}.pt'), map_location=DEVICE))
    p, a = predict(m, LD['adv']['test']); ps_or.append(p); del m
mis = sc(np.mean(ps_or, axis=0), a)
ref28 = json.load(open(os.path.join(RESULTS_DIR, 'v5_round2_delhi.json')))['test']['dyn_adv']
print(f"dyn_adv trained on perfect future wind, scored with perfect wind : RMSE {ref28['rmse']:.2f}   (notebook 28)")
print(f"same models fed the real day-old forecast wind                   : RMSE {mis['rmse']:.2f}   MAE {mis['mae']:.2f}   R2 {mis['r2']:.3f}")
print('-> the gap is the cost of using a real forecast without retraining; the retrained dyn_adv_fc models above close part of it.')

# HOW TO CHECK: the second number is higher than 57.71 (a forecast is worse than perfect wind); by how much is the finding.

dyn_adv trained on perfect future wind, scored with perfect wind : RMSE 57.71   (notebook 28)
same models fed the real day-old forecast wind                   : RMSE 58.32   MAE 38.21   R2 0.665
-> the gap is the cost of using a real forecast without retraining; the retrained dyn_adv_fc models above close part of it.


In [13]:
# results: validation (used to choose) and test (scored once)
def ens(split, act, names): return {n: sc(np.mean(pred[split][n], axis=0), act) for n in names}
SV, ST = ens('val', val_actual, ALL), ens('test', test_actual, ALL)
MIX = {'val': np.mean([np.mean(pred['val'][n], 0) for n in ALL], 0), 'test': np.mean([np.mean(pred['test'][n], 0) for n in ALL], 0)}
SV['mix_all3'], ST['mix_all3'] = sc(MIX['val'], val_actual), sc(MIX['test'], test_actual)
print(f"{'variant':14}{'val RMSE':>10}{'test RMSE':>11}{'test MAE':>10}{'test R2':>9}")
for n in ALL + ['mix_all3']:
    print(f"{n:14}{SV[n]['rmse']:>10.2f}{ST[n]['rmse']:>11.2f}{ST[n]['mae']:>10.2f}{ST[n]['r2']:>9.3f}")

def horizon_rmse(split_pred, act):
    p, a_ = from_log(split_pred), from_log(act)
    return [float(np.sqrt(np.mean((p[:, h] - a_[:, h]) ** 2))) for h in (0, 2, 5, 11, 17, 23)]
anc = np.stack([Yl_full[g + INPUT_WINDOW - 1] for g in test_starts])
print(f"\nTEST RMSE by horizon      {'+1h':>7}{'+3h':>7}{'+6h':>7}{'+12h':>7}{'+18h':>7}{'+24h':>7}")
print(f"{'persistence':24}" + ''.join(f'{x:>7.1f}' for x in horizon_rmse(np.repeat(anc[:, None, :], HORIZON, axis=1), test_actual)))
for n in ALL: print(f'{n:24}' + ''.join(f'{x:>7.1f}' for x in horizon_rmse(np.mean(pred['test'][n], 0), test_actual)))
print(f"{'mix_all3':24}" + ''.join(f'{x:>7.1f}' for x in horizon_rmse(MIX['test'], test_actual)))

print('\nreference scores (test RMSE, from earlier notebooks, 5-seed ensembles):')
print('  perfect future wind : dyn_adv 57.71 | none_nb 58.04 | none_adv 57.91   (notebook 28)')
print('  real day-old forecast, plain models : graph-free 58.53 | static 58.62 | wind graph 59.51   (notebooks 19/20)')
bw = min(['dyn_adv_fc', 'none_adv_fc'], key=lambda n: SV[n]['rmse']); bo = 'none_nb_fc'
print(f"\nchosen on validation among the upwind-feature models: {bw} (val {SV[bw]['rmse']:.2f})")
print(f"TEST: dyn_adv_fc {ST['dyn_adv_fc']['rmse']:.2f} | none_adv_fc {ST['none_adv_fc']['rmse']:.2f} | none_nb_fc {ST['none_nb_fc']['rmse']:.2f}")
print(f"wind graph + upwind vs graph-free + neighbour mean : {ST['none_nb_fc']['rmse'] - ST['dyn_adv_fc']['rmse']:+.2f}   (positive = wind-aware better; under about 1 is inside seed noise)")
json.dump({'val': SV, 'test': ST, 'oracle_models_fed_forecast_wind': mis, 'reference_oracle_dyn_adv_test': ref28,
           'note': 'future-hour wind channels and forecast graph from the day-old forecast; 5-seed ensembles; mix_all3 = plain average of the three'},
          open(os.path.join(RESULTS_DIR, 'v7_realfc_delhi.json'), 'w'), indent=2)
print('saved v7_realfc_delhi.json')

# HOW TO READ: the dyn_adv_fc / none_nb_fc test RMSEs are the numbers a live system can honestly claim. Expect them 0.3-1.5 above the perfect-wind
# figures. If the wind-aware model is within about 1 of the graph-free one, the conclusion of notebooks 27-28 stands.

variant         val RMSE  test RMSE  test MAE  test R2
dyn_adv_fc         55.72      59.20     38.49    0.654
none_adv_fc        55.82      58.11     37.97    0.667
none_nb_fc         55.89      58.30     38.01    0.665
mix_all3           55.68      58.39     38.01    0.664

TEST RMSE by horizon          +1h    +3h    +6h   +12h   +18h   +24h
persistence                33.8   61.2   82.9   93.2   92.6   76.7
dyn_adv_fc                 40.3   48.3   55.8   61.1   63.9   64.0
none_adv_fc                41.5   47.8   54.9   59.8   62.5   62.7
none_nb_fc                 41.0   47.7   55.0   60.0   62.8   63.1
mix_all3                   40.6   47.7   55.1   60.2   62.9   63.2

reference scores (test RMSE, from earlier notebooks, 5-seed ensembles):
  perfect future wind : dyn_adv 57.71 | none_nb 58.04 | none_adv 57.91   (notebook 28)
  real day-old forecast, plain models : graph-free 58.53 | static 58.62 | wind graph 59.51   (notebooks 19/20)

chosen on validation among the upwind-feature mo